# Efemerideen Etenguneen Eragina IRKGL16 Integrazio Metodoan

## Sarrera: Zergatik da Sinkronizazioa Garrantzitsua?

Efemerideen sinkronizazioa teknika erabakigarria da zehaztasun handiko simulazioetan (adibidez, Apophis asteroidearen ibilbidea kalkulatzean), efemerideen datu-egiturak sortutako zarata numerikoa ezabatzeko.

### 1. Arazoa: Efemerideen Izaera Zatikatua

Simulazioetan erabiltzen diren planeten eta ilargiaren posizioak (NASAren SPICE sistema erabiliz) ez dira funtzio analitiko perfektu eta infinituak. Horren ordez, **zatika definitutako polinomioak** (Chebyshev polinomioak) dira:

- Polinomio hauen artean **"josturak" edo mugak** daude (nodoak)
- Integrazio-pausoak $(h)$ muga horiek arbitrarioki zeharkatzen baditu, **interpolazio-errore txikiak** sortzen dira
- Doitasun handiko simulazioetan, errore txiki hauek **pilatu** egiten dira, errore-zoru edo "plateau" bat sortuz

### 2. Irtenbidea: Bi Urratseko Sinkronizazioa

Sinkronizazioak integrazio-pausoak efemerideen egitura matematikoarekin "lerrokatzea" bilatzen du bi modutan:

**a) Pausoaren tamaina $(h)$ egokitzea:**
- Integrazio-pausoa aldatu egiten da, efemerideen polinomio-luzeraren (adibidez, $L_{MOON}$, Ilargiaren polinomioaren iraupena) **zatitzaile zehatza** izan dadin
- Horrek bermatzen du simulazioaren urratsak beti eroriko direla nodoetan edo tarte proportzionaletan

**b) Hasiera puntua $(t_0)$ lerrokatzea:**
- Pausoa egokitzea ez da nahikoa; simulazioa polinomioaren "erdi" batean hasten bada, oraindik mugak zeharka litezke
- Sinkronizazio osoak simulazioaren hasiera efemerideen hurrengo nodo edo muga zehatzera mugitzen du

### 3. Emaitza Esperoak

**Ordena altuko metodoetan (IRKGL16):** $O(h^{16})$ bezalako metodoetan, non doitasuna oso azkar iristen den makina-mailako mugara $(10^{-16})$, sinkronizazioak laguntzen du interpolazio-zarata minimizatzen.

---

## Analisiaren Helburua

Notebook honetan bi fase aztertuko ditugu:

1. **Fase 1:** Apirilaren 30era iritsi (doitasun onargarriarekin)
2. **Fase 2:** Urrats luzeagoekin etenguneen eragina aztertu:
   - Kasu **sinkronizatua**: Urratsak $L_{MOON}$-en zatitzaile zehatzak
   - Kasu **ez-sinkronizatua**: Urrats arbitrarioak
   
Helburu nagusia: **IRKGL16-k bere 16. ordena mantentzen duen** egiaztatzea eta **etenguneen eraginaren neurria** zehaztea.

## 1. Liburutegiak eta Iturburu Kodea Kargatu

In [1]:
using Pkg
Pkg.activate("..") 

# Paketea proiektu gisa kargatu
try
    using GravitationSimulation
catch
    # Ez bada aurkitzen pakete bezala, fitxategia kargatu
    include("../src/GravitationSimulation.jl")
    using .GravitationSimulation
end

# Beste liburutegi guztiak
using LittleEphemeris
using JSON
using CSV
using SPICE
using DataFrames
using FFTW
using Plots
using BenchmarkTools
using LinearAlgebra
using IRKGaussLegendre


  Activating project at `~/Documentos/Ingenieritza_Informatikoa/GrAL/GrAL_github`


In [2]:
furnsh("./data/naif0012.tls", "./data/de440.bsp")

## 2. Hasierako Baldintzak eta Sistema Konfiguratu

Simulazioa 2029ko urtarrilean hasiko dugu, Apophis Lurretik hurbil pasatzeko aurretik.

In [3]:
# Gorputzen koefizienteak kargatu
# 3: Earth-Moon Barycenter, 5: Jupiter Barycenter, 10: Sun, 301: Moon
# 1: Mercury, 2: Venus, 4: Mars, 6: Saturn, 7: Uranus, 8: Neptune
ID_list = [1, 2, 3, 4, 5, 6, 7, 8, 10, 301]

et_0 = 10593.535998938605 * 86400
et_end = et_0 + 120*86400
time_interval = (et_0, et_end)
time_interval_list = fill(time_interval, length(ID_list))

# Koefizienteak sortu (denbora tarte osorako)
create_coeffs_file("./data/coeffs.json", "./data/coeffs.csv", ID_list, time_interval_list)

Mercury = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 1, time_interval);
Venus   = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 2, time_interval);
Earth   = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 3, time_interval);
Mars    = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 4, time_interval);
Jupiter = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 5, time_interval);
Saturn  = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 6, time_interval);
Uranus  = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 7, time_interval);
Neptune = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 8, time_interval);
Sun     = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 10, time_interval);
Moon    = BodyCoeffs("data/coeffs.json", "data/coeffs.csv", 301, time_interval);



In [4]:


# Hasierako balioak eta konstanteak
u0 = [-5.5946538550488512E-01, 8.5647564757574512E-01, 3.0415066217102493E-01,
      -1.3818324735921638E-02, -6.0088275597939191E-03, -2.5805044631309632E-03]

# Heliozentrikotik barizentrikora
eguzkia_pos_barizentrikoa = [0.001232781221250307, -0.0012750764430978325, -0.0005187131180711941]
u0[1:3] += eguzkia_pos_barizentrikoa

# Unit conversion
AU = 149597870.7
DAY = 86400.0
u0[1:3] *= AU
u0[4:6] *= (AU / DAY)

# Grabitazio parametroak (km^3/s^2)
mu_S = 1.32712440042e11
mu_E = 398600.435
mu_M = 4902.80
mu_Mer = 22032.09
mu_V = 324858.59
mu_Ma = 42828.37
mu_J = 1.26686534e8
mu_Sat = 3.7931187e7
mu_U = 5.793939e6
mu_N = 6.836529e6

p_all = [mu_S, mu_E, mu_M, mu_Mer, mu_V, mu_Ma, mu_J, mu_Sat, mu_U, mu_N]
# Lehen agian p_all = [mu1, mu2, ...] zenuen. 
# Orain egitura hau behar du:
p_all = (
    mus = [mu_S, mu_E, mu_M, mu_Mer, mu_V, mu_Ma, mu_J, mu_Sat, mu_U, mu_N], # mu-en zerrenda
    bodies = [
        Sun, Earth, Moon, Mercury,
        Venus, Mars, Jupiter, Saturn, Uranus, Neptune] # posizioen zerrenda
)

t_0 = et_0
t_end = et_end

N_samples = 1000
dt_total = t_end - t_0
dt_out = dt_total / (N_samples - 1)

println("Simulazio parametroak:")
println("Hasiera (ET): $t_0")
println("Bukaera (ET): $t_end")
println("Lagin kopurua (Nomatua): $N_samples")
println("Irteera pausoa (dt_out): $dt_out s (~$(round(dt_out/3600, digits=2)) ordu)")

Simulazio parametroak:
Hasiera (ET): 9.152815103082955e8
Bukaera (ET): 9.256495103082955e8
Lagin kopurua (Nomatua): 1000
Irteera pausoa (dt_out): 10378.378378378378 s (~2.88 ordu)


## 3. FASE 1: Apirilaren 30era Iritsi

Lehenik, simulazioa hasieratik (2029ko urtarrila) Apirilaren 30era eraman behar dugu. Zati honetarako ez dugu muturreko zehaztasunik bilatzen, baina bai abiapuntu fidagarri bat bigarren faserako.

- **Iraupena**: ~120 egun (urtarrilatik apirilera)
- **Integratzailea**: IRKGL16
- **Pausoa**: 3600 segundu (1 orduko urratsak)
- **Mota**: Pauso finkoa (adaptive=false)

In [5]:
# Denbora tartea definitu (Urtarrilak 1 -> Apirilak 30)
# 120 egun inguru: Apophis Lurretik urrundu dela ziurtatzeko
t_start_phase1 = et_0
t_end_phase1 = et_0 + (120 * 86400)  # 120 egun segundutan

println("Simulazioa hasten: $t_start_phase1 -> $t_end_phase1")

# Integrazioa egin (IRKGL16 erabiliz, nahiko segurua da)
# Pauso arrunta erabil dezakegu hemen (ez da sinkronizazioa kritikoa oraindik)
prob_phase1 = ODEProblem(f_all!, u0, (t_start_phase1, t_end_phase1), p_all)
sol_phase1 = solve(prob_phase1, IRKGL16(); dt=3600.0, adaptive=false)

# Bigarren faseko hasiera puntua gorde
u_apirilak30 = sol_phase1.u[end]
t_apirilak30 = sol_phase1.t[end]

println("✓ 1. Fasea amaituta.")
println("Data (ET): $t_apirilak30")
println("Apirilak 30 egoera gordeta.")

Simulazioa hasten: 9.152815103082955e8 -> 9.256495103082955e8
✓ 1. Fasea amaituta.
Data (ET): 9.256495103082955e8
Apirilak 30 egoera gordeta.


## 4. FASE 2: Analisiaren Parametroak Definitu

Orain puntu horretatik aurrera (Apophis espazio sakonean dagoela), urrats oso luzeak probatuko ditugu. Helburua da ikustea ea efemerideen polinomio-mugak (etenguneak) zeharkatzean erroreak nola jokatzen duen.

### Bi Kasu Konparatuko Ditugu:

1. **Sinkronizatua**: Urratsa $(h)$ Ilargiaren efemerideen iraupenaren $(L_{MOON})$ zatitzaile zehatza da
2. **Ez-sinkronizatua**: Urrats arbitrarioa, efemerideen mugekin bat ez datorrena

In [7]:
# Simulazio luzeagoa (adibidez, 200 egun gehiago)
t_end_analysis = t_apirilak30 + (200 * 86400)
prob_analysis = ODEProblem(f_all!, u_apirilak30, (t_apirilak30, t_end_analysis), p_all)

# --- A) PARAMETROEN KALKULUA ---
# Ilargiaren polinomioaren luzera lortu (L_MOON)
coeffs_data = JSON.parsefile("data/coeffs.json")
moon_entry = filter(x -> x["bodyID"] == 301, coeffs_data)[1]
L_MOON = moon_entry["timeIntervals"][2] - moon_entry["timeIntervals"][1] 
# (Normalean 345600.0 s izaten da, ~4 egun)

println("L_MOON (Ilargiaren polinomioaren iraupena): $L_MOON s")
println("Hau da, gutxi gorabehera $(L_MOON/86400) egun\n")

# KASU 1: SINKRONIZATUA (Nodoekin bat)
# Urratsa L_MOON-en zatitzaile zehatza izan dadin
zatitzailea = 8  # Adibidez, 4 egun / 8 = 12 orduko pauso "perfektua"
h_sinkro = L_MOON / zatitzailea

# KASU 2: EZ-SINKRONIZATUA (Arbitrarioa)
# h_sinkro-tik gertu, baina "zikina", mugekin bat ez egiteko
h_ez_sinkro = h_sinkro + 123.456 

println("Konparaketa parametroak:")
println("  - h Sinkronizatua: $h_sinkro s ($(h_sinkro/3600) ordu)")
println("  - h Ez-sinkronizatua: $h_ez_sinkro s ($(h_ez_sinkro/3600) ordu)")

L_MOON (Ilargiaren polinomioaren iraupena): 345600.0 s
Hau da, gutxi gorabehera 4.0 egun

Konparaketa parametroak:
  - h Sinkronizatua: 43200.0 s (12.0 ordu)
  - h Ez-sinkronizatua: 43323.456 s (12.034293333333332 ordu)


## 5. Errore Kalkulu Funtzioa Definitu

Auto-konsistentzia testa egingo dugu: soluzio bat pauso handiarekin kalkulatu, eta beste bat pauso txikiagoarekin (erreferentzia gisa). Bi soluzio hauen arteko diferentzia emango digu trunkatze-errorea.

In [8]:
# Funtzio laguntzailea errorea kalkulatzeko
function kalkulatu_errorea(h_try, steps_ratio)
    """
    Erreferentzia-soluzio bat eta test-soluzio bat konparatzen ditu.
    
    Parametroak:
    - h_try: Probatzeko urrats-tamaina
    - steps_ratio: Zenbat aldiz txikiagoa den erreferentziako pausoa
    
    Itzulketa:
    - Errore erlatiboa (normalizatua)
    """
    # Erreferentzia: Pauso oso txikia (doitasun handia)
    h_ref = h_try / steps_ratio 
    
    # Soluzio "zehatza" (erreferentzia)
    sol_ref = solve(prob_analysis, IRKGL16(); dt=h_ref, adaptive=false, saveat=h_try)
    
    # Soluzioa probatzeko (urrats handia)
    sol_test = solve(prob_analysis, IRKGL16(); dt=h_try, adaptive=false, saveat=h_try)
    
    # Konparatu amaierako posizioa (lehenengo 3 osagaiak = Apophisen posizioa)
    err = norm(sol_test.u[end][1:3] - sol_ref.u[end][1:3]) / norm(sol_ref.u[end][1:3])
    return err
end

println("✓ Errore-kalkulu funtzioa definituta.")

✓ Errore-kalkulu funtzioa definituta.


## 6. Analisi Konparatiboa Exekutatu

Orain, begizta batean, $m$ balio desberdinak probatuko ditugu. $m$ handitu ahala, urrats-tamaina $(h)$ txikitu egiten da. Bi kasuak konparatuko ditugu:

- **Sinkronizatua**: $h = \frac{L_{MOON}}{zatitzailea \cdot m}$ (beti $L_{MOON}$-en zatitzaile zehatza)
- **Ez-sinkronizatua**: $h = \frac{L_{MOON}}{zatitzailea \cdot m} + 10.1$ (desfase txiki bat gehituz)

Horrela ikus dezakegu ea etenguneetan erortzen ez diren urratsek errore handiagoa ote duten.

In [10]:
# Begizta: Urrats desberdinak probatu 16. ordena egiaztatzeko
# m (dibisorea) handitu ahala, h txikitu egiten da.
m_values = [1, 2, 4, 8, 16, 32, 64] 
erroreak_sinkro = []
erroreak_ez_sinkro = []
h_values_sinkro = []
h_values_ez_sinkro = []

println("\n" * "="^60)
println("ANALISIA EXEKUTATZEN...")
println("="^60 * "\n")

# KASU 1: SINKRONIZATUA (h beti L_MOON-en zatitzailea)
println("📊 KASU 1: SINKRONIZATUA")
println("-"^60)
for m in m_values
    h = L_MOON / (zatitzailea * m)  # h txikitzen goaz, baina beti sinkronizatuta
    err = kalkulatu_errorea(h, 4)   # Erref pausoa 4 aldiz txikiagoa
    push!(erroreak_sinkro, err)
    push!(h_values_sinkro, h)
    println("  m=$m → h=$(round(h, digits=2))s → Errore=$err")
end

println("\n📊 KASU 2: EZ-SINKRONIZATUA")
println("-"^60)
# KASU 2: EZ-SINKRONIZATUA
for m in m_values
    h = (L_MOON / (zatitzailea * m)) + 10.1  # Desfase txiki bat beti
    err = kalkulatu_errorea(h, 4)
    push!(erroreak_ez_sinkro, err)
    push!(h_values_ez_sinkro, h)
    println("  m=$m → h=$(round(h, digits=2))s → Errore=$err")
end

println("\n" * "="^60)
println("✓ ANALISIA AMAITUTA")
println("="^60)


ANALISIA EXEKUTATZEN...

📊 KASU 1: SINKRONIZATUA
------------------------------------------------------------


LoadError: BoundsError: attempt to access 32-element Vector{Float64} at index [33]

## 7. Emaitzak Aztertu eta Grafikoak Sortu

### 7.1. Log-Log Grafikoa

Errorearen eboluzioa urrats-tamainaren arabera ikusiko dugu. Grafiko log-log batean, $O(h^{16})$ portaera lerro zuzen bat izan beharko litzateke, 16-ko maldarekin.

In [11]:
# --- GRAFIKOA ETA 16. ORDENA EGIAZTATZEA ---

p = plot(
    xaxis=:log, yaxis=:log, 
    title="IRKGL16: Etenguneen Eragina Urrats Luzeetan", 
    xlabel="h (segundu)", 
    ylabel="Errore Erlatiboa",
    legend=:bottomright,
    size=(800, 600),
    grid=true
)

# Bi kurbak marraztu
plot!(p, h_values_sinkro, erroreak_sinkro, 
      label="Sinkronizatua (Etenguneetan)", 
      marker=:circle, markersize=6, linewidth=2, color=:blue)

plot!(p, h_values_ez_sinkro, erroreak_ez_sinkro, 
      label="Ez-sinkronizatua", 
      marker=:square, markersize=6, linewidth=2, color=:red)

# O(h^16) malda teorikoa gehitu
# Lehen puntutik abiatuta lerro teorikoa marraztu
c = erroreak_sinkro[1] / (h_values_sinkro[1]^16)
y_theoretical = [c * h^16 for h in h_values_sinkro]
plot!(p, h_values_sinkro, y_theoretical, 
      label="O(h¹⁶) Teorikoa", 
      linestyle=:dash, linewidth=2, color=:green)

display(p)

LoadError: BoundsError: attempt to access 0-element Vector{Any} at index [1]

### 7.2. Konbergentzia Maldaren Kalkulua

Lerrokadura linealarekin (logaritmikoki), zenbateko malda duten bi kurbek kalkulatu dezakegu. Espero den balioa $\approx 16$ da IRKGL16 metodoarentzat.

In [ ]:
# Malda kalkulatu (Slope analysis - Lerro-egokitzapen lineala)

# KASU SINKRONIZATUA
log_h_sinkro = log.(h_values_sinkro)
log_err_sinkro = log.(erroreak_sinkro)
X_sinkro = hcat(ones(length(log_h_sinkro)), log_h_sinkro)
coeffs_sinkro = X_sinkro \ log_err_sinkro
malda_sinkro = coeffs_sinkro[2]

# KASU EZ-SINKRONIZATUA
log_h_ez_sinkro = log.(h_values_ez_sinkro)
log_err_ez_sinkro = log.(erroreak_ez_sinkro)
X_ez_sinkro = hcat(ones(length(log_h_ez_sinkro)), log_h_ez_sinkro)
coeffs_ez_sinkro = X_ez_sinkro \ log_err_ez_sinkro
malda_ez_sinkro = coeffs_ez_sinkro[2]

println("\n" * "="^60)
println("KONBERGENTZIA ORDENA (Slope Analysis)")
println("="^60)
println("Lortutako malda (Sinkronizatua):     $(round(malda_sinkro, digits=2))")
println("Lortutako malda (Ez-sinkronizatua):  $(round(malda_ez_sinkro, digits=2))")
println("Espero den balioa teorikoa:          ~16.0")
println("="^60)

if abs(malda_sinkro - 16.0) < 2.0
    println("✓ Sinkronizatua: 16. ordena mantentzen da!")
else
    println("⚠ Sinkronizatua: Desbideratze nabarmena 16. ordenetik")
end

if abs(malda_ez_sinkro - 16.0) < 2.0
    println("✓ Ez-sinkronizatua: 16. ordena mantentzen da!")
else
    println("⚠ Ez-sinkronizatua: Desbideratze nabarmena 16. ordenetik")
end

## 8. Ondorioak eta Interpretazioa

### Zer espero dugu emaitzetan?

Analisi hau iturri zientifikoen arabera eginda, emaitza hauek bilatu beharko genituzke:

#### 1. **Ordenaren Mantentzea**

IRKGL16 metodoa oso sendoa da. 16. ordena $(O(h^{16}))$ mantendu beharko luke bi kasuetan hasieran. Hau da, grafikoan **bi kurbek malda oso pikoa** izan beharko lukete (log-log eskalan).

#### 2. **Etenguneen Eragina (Errore Zorua)**

##### **Kasu Ez-sinkronizatua** ⚠️
- Doitasun maila oso altuetara iristean (errore oso txikiak), urratsa etengune baten gainean erortzen bada, **interpolazio-errore txiki bat** ("zarata") sartuko da
- Honek $h^{16}$ konbergentzia **"apurtu" dezake** puntu jakin batetik aurrera
- Erroreak beherantz egiteari utziko dio (**plateau edo zorua**)

##### **Kasu Sinkronizatua** ✅
- Urratsak beti nodoekin bat datozenez, efemerideen interpolazioa **"perfektua"** da (edo gutxienez, koherentea)
- Kasu honetan, erroreak ordenagailuaren doitasun mugara $(10^{-16})$ iritsi arte **jarraitu beharko luke jaisten**
- Teorian **portaera hobea** erakutsiko du

### Ondorio Praktikoa

Kode honekin **garbi ikusiko dugu** ea urrats luzeetan (non errorearen iturri nagusia oraindik integratzailea den eta ez efemerideak) **sinkronizazioak merezi duen** ala ez. 

Urrats handiekin lan egitean (esaterako, egun batzuetako pausoekin), etenguneekiko sinkronizazioak eragin nabarmena izan dezake simulazioaren fidagarritasunean.